# DDRI cluster01 subset(축소 피처 조합) 실험

이 노트북은 `cluster01(아침 도착 업무 집중형)`에 대해 설계된 핵심 피처 subset(축소 피처 조합)을 비교하고, 현재 3차 최적화 결과보다 더 적은 피처(feature, 입력 변수)로 유지 또는 개선이 가능한지 확인하는 문서다.


## 용어 설명
- `feature(입력 변수)`: 모델이 예측에 사용하는 입력 값
- `lag(과거 시점 값)`: 같은 대여소의 몇 시간 전 대여량을 가져온 피처
- `rolling(이동 통계)`: 최근 몇 시간 대여량의 평균이나 표준편차를 계산한 피처
- `Train / Validation / Test(학습 / 검증 / 최종 평가)`: 모델을 학습하고 비교하고 마지막으로 점검하는 데이터 구간
- `objective(학습 목표 함수)`: 모델이 어떤 방식으로 오차를 줄이도록 학습할지 정하는 기준
- `RMSE(제곱평균제곱근오차)`: 큰 오차에 더 민감한 대표 회귀 평가 지표
- `MAE(평균절대오차)`: 실제값과 예측값의 절대 차이 평균
- `R²(설명력)`: 모델이 변동을 얼마나 설명하는지 보는 지표
- `subset(축소 피처 조합)`: 전체 피처 중 일부만 골라 만든 비교용 조합


## 1. 실험 목적(Experiment Goal, 실험 목표)

- 담당 군집: `여기에 군집명 기입`
- 목표: 대표 대여소 3개를 대상으로 `2023 train(학습) / 2024 validation(검증) / 2025 test(최종 평가)` 정책에 따라 공통 모델 실험을 수행한다.
- 필수 모델: `LinearRegression(선형회귀 모델)`, `LightGBM_RMSE(RMSE 기준 LightGBM)`
- 필수 평가 지표: `RMSE(제곱평균제곱근오차)`, `MAE(평균절대오차)`, `R²(설명력)`


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    from lightgbm import LGBMRegressor
except ImportError:
    LGBMRegressor = None

BASE_DIR = Path('/Users/cheng80/Desktop/ddri_work/works/05_prediction_long')
TRAIN_PATH = Path('/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소_예측데이터_15개/second_round_data/ddri_prediction_long_train_2023_2024_second_round_feature_collection.csv')
TEST_PATH = Path('/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소_예측데이터_15개/second_round_data/ddri_prediction_long_test_2025_second_round_feature_collection.csv')
OUTPUT_DIR = BASE_DIR / 'output' / 'team_cluster_subset_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_STATION_GROUP = '아침 도착 업무 집중형'  # 담당 군집 고정
DISPLAY_NAME = TARGET_STATION_GROUP

RANDOM_STATE = 42


## 2. 입력 데이터 로드(Input Data Load, 입력 데이터 불러오기)

- 공통 입력 정본은 `3조 공유폴더/대표대여소_예측데이터_15개/raw_data/` 아래 CSV 2개다.
- 개인이 별도 가공한 CSV를 정본처럼 사용하지 않는다.


In [2]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

print('train shape:', train_raw.shape)
print('test shape:', test_raw.shape)
print('station groups:', sorted(train_raw['station_group'].dropna().unique().tolist()))


train shape: (263160, 51)
test shape: (131400, 51)
station groups: ['생활권 혼합형', '아침 도착 업무 집중형', '업무/상업 혼합형', '외곽 주거형', '주거 도착형']


## 3. 담당 군집 필터링(Filter Assigned Cluster, 담당 군집 추리기)

- 반드시 `station_group(담당 군집 그룹명)` 1개만 남긴다.
- 결과적으로 대표 대여소 3개만 남아야 한다.


In [3]:
train_group = train_raw.loc[train_raw['station_group'] == TARGET_STATION_GROUP].copy()
test_group = test_raw.loc[test_raw['station_group'] == TARGET_STATION_GROUP].copy()

print('train_group shape:', train_group.shape)
print('test_group shape:', test_group.shape)
print(train_group[['station_id', 'station_name']].drop_duplicates().sort_values('station_id').to_string(index=False))


train_group shape: (52632, 51)
test_group shape: (26280, 51)
 station_id   station_name
       2323  주식회사 오뚜기 정문 앞
       2348   포스코사거리(기업은행)
       2377       수서역 5번출구


## 4. 전처리 및 파생 피처 생성(Preprocessing And Feature Engineering, 전처리와 파생 피처 생성)

공통 프로토콜:

- `date(날짜)`는 날짜형으로 변환한다.
- `station_id(숫자 대여소 ID)`, `date(날짜)`, `hour(시간)` 기준으로 정렬한다.
- `lag_1h(1시간 전 대여량)`, `lag_24h(24시간 전 대여량)`, `lag_168h(168시간 전 대여량)`를 만든다.
- `rolling_mean_24h(최근 24시간 대여량 이동평균)`, `rolling_std_24h(최근 24시간 대여량 이동표준편차)`, `rolling_mean_168h(최근 168시간 대여량 이동평균)`, `rolling_std_168h(최근 168시간 대여량 이동표준편차)`를 만든다.
- rolling(이동 통계)은 반드시 `station_id(숫자 대여소 ID)`별로 계산하고, `shift(1)(직전 시점으로 한 칸 미루기)` 후 rolling을 적용한다.
- `lag_168h(168시간 전 대여량)`는 `1주 전 동일 요일·동일 시각 대여량`이다.


In [4]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result['date'] = pd.to_datetime(result['date'])
    result = result.sort_values(['station_id', 'date', 'hour']).reset_index(drop=True)

    # 1차/2차 공통 lag
    group = result.groupby('station_id', group_keys=False)['rental_count']
    result['lag_1h'] = group.shift(1)
    result['lag_24h'] = group.shift(24)
    result['lag_168h'] = group.shift(168)

    shifted = group.shift(1)
    result['rolling_mean_24h'] = shifted.rolling(24).mean().reset_index(level=0, drop=True)
    result['rolling_std_24h'] = shifted.rolling(24).std().reset_index(level=0, drop=True)
    result['rolling_mean_168h'] = shifted.rolling(168).mean().reset_index(level=0, drop=True)
    result['rolling_std_168h'] = shifted.rolling(168).std().reset_index(level=0, drop=True)

    # 3차 추가 단기 추세
    result['lag_48h'] = group.shift(48)
    result['rolling_mean_6h'] = shifted.rolling(6).mean().reset_index(level=0, drop=True)
    result['rolling_std_6h'] = shifted.rolling(6).std().reset_index(level=0, drop=True)
    result['rolling_mean_12h'] = shifted.rolling(12).mean().reset_index(level=0, drop=True)
    result['rolling_std_12h'] = shifted.rolling(12).std().reset_index(level=0, drop=True)

    # 출근형 세분 피처
    result['commute_morning_flag'] = result['hour'].isin([7, 8, 9]).astype(int)
    result['commute_evening_flag'] = result['hour'].isin([17, 18, 19]).astype(int)
    return result

train_feat = build_features(train_group)
test_feat = build_features(test_group)

train_feat[['station_id', 'date', 'hour', 'rental_count', 'lag_1h', 'lag_24h', 'lag_168h']].head()


,station_id,date,hour,rental_count,lag_1h,lag_24h,lag_168h
0,2323,2023-01-01,0,0,NaN,NaN,NaN
1,2323,2023-01-01,1,0,0.0,NaN,NaN
2,2323,2023-01-01,2,0,0.0,NaN,NaN
3,2323,2023-01-01,3,0,0.0,NaN,NaN
4,2323,2023-01-01,4,0,0.0,NaN,NaN


## 5. 시간 분할 정책(Time-Based Split Policy, 시간 기준 분할 정책)

- 학습(`Train`, 학습 구간): `2023`
- 검증(`Validation`, 검증 구간): `2024`
- 테스트(`Test`, 최종 평가 구간): `2025`

랜덤 분할은 금지한다.


In [5]:
train_2023 = train_feat.loc[train_feat['date'].dt.year == 2023].copy()
valid_2024 = train_feat.loc[train_feat['date'].dt.year == 2024].copy()
test_2025 = test_feat.copy()

print('train_2023:', train_2023.shape)
print('valid_2024:', valid_2024.shape)
print('test_2025:', test_2025.shape)


train_2023: (26280, 62)
valid_2024: (26352, 62)
test_2025: (26280, 62)


## 6. 입력 피처 목록(Input Feature List, 입력 피처 목록)

아래 목록은 공통 기본 피처(feature, 입력 변수)다. 개인 실험에서 피처를 추가하거나 제거했다면 반드시 이유를 마크다운에 남긴다.


In [6]:
target_col = 'rental_count'

categorical_base_features = [
    'station_id',
    'cluster',
    'mapped_dong_code',
]

numeric_base_features = [
    'hour', 'weekday', 'month', 'holiday',
    'temperature', 'humidity', 'precipitation', 'wind_speed',
    'lag_1h', 'lag_24h', 'lag_168h',
    'rolling_mean_24h', 'rolling_std_24h',
    'rolling_mean_168h', 'rolling_std_168h',
    'is_weekend', 'is_rainy', 'hour_sin', 'hour_cos',
]

subset_feature_map = {
    'baseline_full': [
        'lag_48h',
        'rolling_mean_6h', 'rolling_std_6h',
        'is_commute_hour', 'is_after_holiday',
        'subway_distance_m', 'bus_stop_count_300m',
        'commute_morning_flag', 'commute_evening_flag',
        'rolling_mean_12h', 'rolling_std_12h',
    ],
    'subset_a_commute_transit': [
        'is_commute_hour', 'commute_morning_flag', 'commute_evening_flag',
        'subway_distance_m', 'bus_stop_count_300m',
    ],
    'subset_b_commute_transit_short_trend': [
        'is_commute_hour', 'commute_morning_flag', 'commute_evening_flag',
        'subway_distance_m', 'bus_stop_count_300m',
        'rolling_mean_6h', 'rolling_std_6h', 'lag_48h',
    ],
    'subset_c_current_best_compact': [
        'is_commute_hour', 'commute_morning_flag', 'commute_evening_flag',
        'is_after_holiday',
        'subway_distance_m', 'bus_stop_count_300m',
        'rolling_mean_6h', 'rolling_mean_12h',
        'rolling_std_6h', 'rolling_std_12h', 'lag_48h',
    ],
}

def get_feature_columns(subset_name: str):
    numeric_features = numeric_base_features + subset_feature_map[subset_name]
    feature_cols = categorical_base_features + numeric_features
    return feature_cols, categorical_base_features, numeric_features

pd.DataFrame([
    {
        'subset_name': subset_name,
        'feature_count': len(get_feature_columns(subset_name)[0]),
        'extra_feature_count': len(extra_features),
        'extra_features': ', '.join(extra_features),
    }
    for subset_name, extra_features in subset_feature_map.items()
])


,subset_name,feature_count,extra_feature_count,extra_features
0,baseline_full,33,11,"lag_48h, rolling_mean_6h, rolling_std_6h, is_c..."
1,subset_a_commute_transit,27,5,"is_commute_hour, commute_morning_flag, commute..."
2,subset_b_commute_transit_short_trend,30,8,"is_commute_hour, commute_morning_flag, commute..."
3,subset_c_current_best_compact,33,11,"is_commute_hour, commute_morning_flag, commute..."


## 7. 평가 함수(Evaluation Function, 평가 함수)

모든 팀원은 아래 3개 점수를 동일하게 기록한다.

- `RMSE(제곱평균제곱근오차)`
- `MAE(평균절대오차)`
- `R²(설명력)`


In [7]:
def evaluate_regression(y_true, y_pred):
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
    }

def build_linear_pipeline():
    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
            ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_features),
        ]
    )
    return Pipeline([
        ('preprocessor', preprocessor),
        ('model', LinearRegression()),
    ])

def build_lgbm_model(objective='regression'):
    if LGBMRegressor is None:
        raise ImportError('lightgbm 패키지가 설치되어 있어야 합니다.')
    return LGBMRegressor(
        objective=objective,
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
    )


## 8. subset별 모델 학습(Model Training By Subset, 축소 피처 조합별 모델 학습)

이번 노트북은 subset(축소 피처 조합)별로 `LightGBM_RMSE(RMSE 기준 LightGBM)`와 `LightGBM_Poisson(Poisson 기준 LightGBM)`을 함께 비교한다.

validation(검증) 결과를 기준으로 `subset(축소 피처 조합) + objective(학습 목표 함수)` 조합 중 우세 후보를 선택한다.


In [8]:
results = []

y_train = train_2023[target_col]
y_valid = valid_2024[target_col]

for subset_name in subset_feature_map:
    feature_cols, categorical_features, numeric_features = get_feature_columns(subset_name)
    X_train = train_2023[feature_cols]
    X_valid = valid_2024[feature_cols]

    if LGBMRegressor is not None:
        for objective, name in [('regression', 'LightGBM_RMSE'), ('poisson', 'LightGBM_Poisson')]:
            lgbm_model = build_lgbm_model(objective=objective)
            lgbm_model.fit(X_train, y_train)
            lgbm_pred = lgbm_model.predict(X_valid)
            lgbm_metrics = evaluate_regression(y_valid, lgbm_pred)
            results.append({
                'station_group': TARGET_STATION_GROUP,
                'subset_name': subset_name,
                'feature_count': len(feature_cols),
                'model': name,
                'split': 'validation_2024',
                **lgbm_metrics,
            })

validation_results = pd.DataFrame(results).sort_values(['rmse', 'mae']).reset_index(drop=True)
validation_results


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001350 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2470
[LightGBM] [Info] Number of data points in the train set: 26280, number of used features: 32
[LightGBM] [Info] Start training from score 1.423630


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000906 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2470
[LightGBM] [Info] Number of data points in the train set: 26280, number of used features: 32
[LightGBM] [Info] Start training from score 0.353210


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000910 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1769
[LightGBM] [Info] Number of data points in the train set: 26280, number of used features: 26
[LightGBM] [Info] Start training from score 1.423630


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000899 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1769
[LightGBM] [Info] Number of data points in the train set: 26280, number of used features: 26
[LightGBM] [Info] Start training from score 0.353210


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001003 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2122
[LightGBM] [Info] Number of data points in the train set: 26280, number of used features: 29
[LightGBM] [Info] Start training from score 1.423630


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000886 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2122
[LightGBM] [Info] Number of data points in the train set: 26280, number of used features: 29
[LightGBM] [Info] Start training from score 0.353210


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000982 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2470
[LightGBM] [Info] Number of data points in the train set: 26280, number of used features: 32
[LightGBM] [Info] Start training from score 1.423630


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001091 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2470
[LightGBM] [Info] Number of data points in the train set: 26280, number of used features: 32
[LightGBM] [Info] Start training from score 0.353210


,station_group,subset_name,feature_count,model,split,rmse,mae,r2
0,아침 도착 업무 집중형,subset_a_commute_transit,27,LightGBM_Poisson,validation_2024,1.542137,0.902531,0.664697
1,아침 도착 업무 집중형,subset_b_commute_transit_short_trend,30,LightGBM_Poisson,validation_2024,1.542445,0.900707,0.664563
2,아침 도착 업무 집중형,subset_c_current_best_compact,33,LightGBM_Poisson,validation_2024,1.546241,0.901462,0.662909
3,아침 도착 업무 집중형,baseline_full,33,LightGBM_Poisson,validation_2024,1.547168,0.902801,0.662505
4,아침 도착 업무 집중형,subset_b_commute_transit_short_trend,30,LightGBM_RMSE,validation_2024,1.563580,0.922155,0.655307
5,아침 도착 업무 집중형,subset_a_commute_transit,27,LightGBM_RMSE,validation_2024,1.569092,0.927630,0.652872
6,아침 도착 업무 집중형,subset_c_current_best_compact,33,LightGBM_RMSE,validation_2024,1.570468,0.924950,0.652263
7,아침 도착 업무 집중형,baseline_full,33,LightGBM_RMSE,validation_2024,1.571956,0.925488,0.651604


## 9. 우세 subset 및 모델 선택 후 재학습(Refit Best Subset Model, 우세 축소 피처 조합 재학습)

- validation(검증) 결과를 기준으로 우세 `subset(축소 피처 조합) + 모델` 조합을 선택한다.
- 선택 조합을 `2023 + 2024`로 재학습한다.
- 그 다음 `2025` 테스트셋 점수를 산출한다.


In [9]:
best_row = validation_results.sort_values(['rmse', 'mae']).iloc[0]
best_model_name = best_row['model']
best_subset_name = best_row['subset_name']
best_feature_cols, categorical_features, numeric_features = get_feature_columns(best_subset_name)
print('best_model_name:', best_model_name)
print('best_subset_name:', best_subset_name)

refit_train = train_feat.copy()
X_refit = refit_train[best_feature_cols]
y_refit = refit_train[target_col]
X_test = test_2025[best_feature_cols]
y_test = test_2025[target_col]

if best_model_name == 'LightGBM_Poisson':
    best_model = build_lgbm_model(objective='poisson')
else:
    best_model = build_lgbm_model(objective='regression')

best_model.fit(X_refit, y_refit)
test_pred = best_model.predict(X_test)
test_metrics = evaluate_regression(y_test, test_pred)

test_results = pd.DataFrame([{ 
    'station_group': TARGET_STATION_GROUP,
    'subset_name': best_subset_name,
    'feature_count': len(best_feature_cols),
    'model': best_model_name,
    'split': 'test_2025_refit',
    **test_metrics,
}])
test_results


best_model_name: LightGBM_Poisson
best_subset_name: subset_a_commute_transit
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001199 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1807
[LightGBM] [Info] Number of data points in the train set: 52632, number of used features: 26
[LightGBM] [Info] Start training from score 0.367541


,station_group,subset_name,feature_count,model,split,rmse,mae,r2
0,아침 도착 업무 집중형,subset_a_commute_transit,27,LightGBM_Poisson,test_2025_refit,1.310846,0.771079,0.658481


## 10. 결과 저장(Result Save, 결과 저장)

- 결과 CSV는 담당 군집명이 보이게 저장한다.
- 최소 2개 split(데이터 구간)인 `validation_2024(2024 검증 점수)`, `test_2025_refit(2025 재학습 후 최종 점수)`이 모두 있어야 한다.


In [10]:
final_metrics = pd.concat([validation_results, test_results], ignore_index=True)
safe_group_name = TARGET_STATION_GROUP.replace('/', '_').replace(' ', '_')
metrics_path = OUTPUT_DIR / f'ddri_{safe_group_name}_subset_experiment_model_metrics.csv'
final_metrics.to_csv(metrics_path, index=False)
print('saved:', metrics_path)
final_metrics


saved: /Users/cheng80/Desktop/ddri_work/works/05_prediction_long/output/team_cluster_subset_outputs/ddri_아침_도착_업무_집중형_subset_experiment_model_metrics.csv


,station_group,subset_name,feature_count,model,split,rmse,mae,r2
0,아침 도착 업무 집중형,subset_a_commute_transit,27,LightGBM_Poisson,validation_2024,1.542137,0.902531,0.664697
1,아침 도착 업무 집중형,subset_b_commute_transit_short_trend,30,LightGBM_Poisson,validation_2024,1.542445,0.900707,0.664563
2,아침 도착 업무 집중형,subset_c_current_best_compact,33,LightGBM_Poisson,validation_2024,1.546241,0.901462,0.662909
3,아침 도착 업무 집중형,baseline_full,33,LightGBM_Poisson,validation_2024,1.547168,0.902801,0.662505
4,아침 도착 업무 집중형,subset_b_commute_transit_short_trend,30,LightGBM_RMSE,validation_2024,1.563580,0.922155,0.655307
5,아침 도착 업무 집중형,subset_a_commute_transit,27,LightGBM_RMSE,validation_2024,1.569092,0.927630,0.652872
6,아침 도착 업무 집중형,subset_c_current_best_compact,33,LightGBM_RMSE,validation_2024,1.570468,0.924950,0.652263
7,아침 도착 업무 집중형,baseline_full,33,LightGBM_RMSE,validation_2024,1.571956,0.925488,0.651604
8,아침 도착 업무 집중형,subset_a_commute_transit,27,LightGBM_Poisson,test_2025_refit,1.310846,0.771079,0.658481


## 11. 결과 해석(Result Interpretation, 결과 해석)

아래 항목을 팀원별로 반드시 직접 작성한다.

- validation(검증) 기준 우세 `subset(축소 피처 조합) + 모델` 조합은 무엇인가
- test(최종 평가) 점수는 기존 3차(`RMSE 1.3189`) 대비 어떤가
- 가장 적은 피처(feature, 입력 변수)로 유지 가능한 subset(축소 피처 조합)은 무엇인가
- `cluster01`에서 핵심으로 남겨야 할 피처(feature, 입력 변수)는 무엇인가
- 다음에 버려도 되는 피처(feature, 입력 변수)는 무엇인가


## 12. 제출 전 체크리스트(Submission Checklist, 제출 전 점검표)

- [ ] 담당 군집 3개 대여소만 사용했는가
- [ ] `2023 train(학습) / 2024 validation(검증) / 2025 test(최종 평가)`를 지켰는가
- [ ] `baseline_full(전체 기준선 조합)`, `subset_a(조합 A)`, `subset_b(조합 B)`, `subset_c(조합 C)`를 모두 비교했는가
- [ ] `LightGBM_RMSE(RMSE 기준 LightGBM)`, `LightGBM_Poisson(Poisson 기준 LightGBM)`를 모두 비교했는가
- [ ] `RMSE(제곱평균제곱근오차)`, `MAE(평균절대오차)`, `R²(설명력)`를 validation/test 모두 기록했는가
- [ ] 테스트셋으로 subset(축소 피처 조합)을 직접 고르지 않았는가
- [ ] 최종 채택 subset(축소 피처 조합)과 제외 subset(축소 피처 조합)의 이유를 적었는가
